# 🛡️ Guardrails with LangChain — Crash Course

Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

Before the agent starts (input guardrails)
After it completes (output guardrails)
Around model and tool calls
Common Use Cases:
Use Case	Example
PII leakage prevention	Redact emails/credit cards before logging
Prompt injection blocking	Detect adversarial inputs
Harmful content filtering	Block dangerous requests
Business rule enforcement	Require approval for financial ops
Output quality validation	Ensure response meets safety standards

# ⚖️ Section 2: Two Approaches to Guardrails
### Deterministic Guardrails
- Rule-based: regex, keyword matching, explicit checks
- ✅ Fast, predictable, cost-effective
- ❌ May miss nuanced violations

### Model-Based Guardrails
- Uses LLMs/classifiers for semantic understanding
- ✅ Catches subtle/nuanced issues
- ❌ Slower and more expensive

In [3]:
from dotenv import load_dotenv
load_dotenv()
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")



In [4]:
import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")

for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [5]:
from langchain_groq import ChatGroq

def model_based_guardrail(text:str)->str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"],model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024)
    prompt = f"""Is the following user input safe to process? Reply with only 'SAFE' or 'UNSAFE'. Input: {text}"""

    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
✅ SAFE: Explain how malware spreads


---
## 🔒 Section 3: Built-in Guardrail — PII Detection Middleware

LangChain provides built-in `PIIMiddleware` for detecting and handling **Personally Identifiable Information (PII)**.

### Supported PII Types:
| Type | Example |
|---|---|
| `email` | user@example.com |
| `credit_card` | 5105-1051-0510-5100 |
| `ip` | 192.168.1.1 |
| `mac_address` | 00:1A:2B:3C:4D:5E |
| `url` | https://secret-site.com |

### Strategies:
| Strategy | Result |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-1234` |
| `hash` | `a8f5f167...` |
| `block` | Raises an exception |

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

@tool
def customer_lookup(query:str)->str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

llm = ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"],model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024)

# Create agent with PII Middleware
agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ]
)

print("Agent with PII middleware created successfully!")


Agent with PII middleware created successfully!


In [18]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I'm sorry, but I can't help with that. I don't have access to your email or card information, and I shouldn't be handling sensitive financial details like card numbers.

If you need help with an account or transaction, I'd recommend:

1. **Contact your bank or card issuer directly** through their official app, website, or customer service phone number
2. **Use secure channels** - never share full card numbers, CVV codes, or passwords in chat
3. **Verify the source** - if someone asked you for this information, be cautious as it could be a scam

Is there something else I can help you with that doesn't involve sensitive personal or financial information?


In [22]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='cc28b7d1-9efd-4e75-856e-39cc6c371147'),
  AIMessage(content="I'm sorry, but I can't help with that. I don't have access to your email or card information, and I shouldn't be handling sensitive financial details like card numbers.\n\nIf you need help with an account or transaction, I'd recommend:\n\n1. **Contact your bank or card issuer directly** through their official app, website, or customer service phone number\n2. **Use secure channels** - never share full card numbers, CVV codes, or passwords in chat\n3. **Verify the source** - if someone asked you for this information, be cautious as it could be a scam\n\nIs there something else I can help you with that doesn't involve sensitive personal or financial information?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 

In [25]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [24]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='cc28b7d1-9efd-4e75-856e-39cc6c371147'),
  AIMessage(content="I'm sorry, but I can't help with that. I don't have access to your email or card information, and I shouldn't be handling sensitive financial details like card numbers.\n\nIf you need help with an account or transaction, I'd recommend:\n\n1. **Contact your bank or card issuer directly** through their official app, website, or customer service phone number\n2. **Use secure channels** - never share full card numbers, CVV codes, or passwords in chat\n3. **Verify the source** - if someone asked you for this information, be cautious as it could be a scam\n\nIs there something else I can help you with that doesn't involve sensitive personal or financial information?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 

---
## 👤 Section 4: Built-in Guardrail — Human-in-the-Loop Middleware

Pauses agent execution before sensitive operations and waits for human approval.

**Best for:**
- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

**Key requirement:** A `checkpointer` for state persistence across interrupts.

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query:str)->str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to:str,subject:str,body:str)->str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
agent = create_agent(
    model=llm,
    tools=[search_web,send_email,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":True,
                "delete_records":True,
                "search_web":False
            }
        )
    ],
    checkpointer=InMemorySaver()
)

print("Human-in-the-Loop agent created!")


Human-in-the-Loop agent created!


In [27]:
config = { "configurable":{"thread_id": "session_001"}}
result = agent.invoke(
    {"messages":[{"role":"user","content":"Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='3601f1b2-ccaf-47a0-a07e-b51e318474db'), AIMessage(content="I'd be happy to help send an email about the Q4 results, but I need a bit more information to draft it properly. Could you please provide:\n\n1. **Subject line** (or should I suggest one?)\n2. **Key points or content** to include in the email — for example:\n   - Revenue figures or growth percentages\n   - Notable achievements or highlights\n   - Any challenges or areas for improvement\n   - Key takeaways or next steps\n3. **Tone** — formal, casual, celebratory, etc.\n4. **Any specific details** like dates, product names, or team members to mention\n\nOnce you share these details, I'll draft the email and send it to team@company.com.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 431, 'tot

In [28]:
approved_result = agent.invoke(
    Command(resume={"decision":[{"type":"approve"}]}),
    config=config
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
I'd be happy to help send an email about the Q4 results, but I need a bit more information to draft it properly. Could you please provide:

1. **Subject line** (or should I suggest one?)
2. **Key points or content** to include in the email — for example:
   - Revenue figures or growth percentages
   - Notable achievements or highlights
   - Any challenges or areas for improvement
   - Key takeaways or next steps
3. **Tone** — formal, casual, celebratory, etc.
4. **Any specific details** like dates, product names, or team members to mention

Once you share these details, I'll draft the email and send it to team@company.com.


In [29]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I've stopped the deletion as you requested. Let me know how you'd like to proceed — for example, you might want to:

- Review which records would be affected first (e.g., a SELECT query)
- Soft-delete instead (set a flag) rather than hard-delete
- Narrow the condition (e.g., only inactive users older than a certain date)

Just let me know how you'd like to handle it.
</think>

I've stopped the deletion as you requested. Let me know how you'd like to proceed — for example, you might want to:

- Review which records would be affected first (e.g., a SELECT query)
- Soft-delete instead (set a flag) rather than hard-delete
- Narrow the condition (e.g., only inactive users older than a certain date)

Just let me know how you'd like to handle it.


---
## ⚙️ Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)

Use `before_agent()` to validate or block requests **before any LLM processing begins**.

**Best for:**
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [24]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool


class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state: AgentState,runtime: Runtime) -> dict[str,Any] | None:
        if not state["messages"]:
            return None

        first_messsage = state["messages"][0]
        if first_messsage.type != "human":
            return None

        content = first_messsage.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

# Create agent with content filter
filtered_agent = create_agent(
    model=llm,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"])
    ]
)
print("Content filter agent created!")


Content filter agent created!


In [6]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
**Machine learning (ML)** is a branch of artificial intelligence (AI) that focuses on building systems that can learn from data. Instead of being explicitly programmed with rules for every possible scenario, machine learning algorithms use statistical techniques to identify patterns in data and make predictions or decisions.

### Key Characteristics:
1. **Data-Driven**: ML models are trained on large datasets to recognize patterns.
2. **Adaptive**: The system improves its performance over time as it is exposed to more data.
3. **Predictive**: It can forecast outcomes, classify data, or detect anomalies.

### Common Types of Machine Learning:
- **Supervised Learning**: The model is trained on labeled data (input-output pairs). Examples: spam detection, image classification.
- **Unsupervised Learning**: The model finds hidden patterns in unlabeled data. Examples: customer segmentation, anomaly detection.
- **Reinforcement Learning**: The model learns by interacti

In [7]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


---
## 🔍 Section 6: Custom Guardrail — After-Agent Hook (Output Safety)

Use `after_agent()` to validate the final agent response **before the user sees it**.

**Best for:**
- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through


In [17]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool



class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = self.safety_model = ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"],model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"],model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024),
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [20]:
result = safe_agent.invoke({
    "messages":[{"role":"user","content":"How to hack a server"}]
})
print("response ===>")
print(result)

response ===>
{'messages': [HumanMessage(content='How to hack a server', additional_kwargs={}, response_metadata={}, id='e208f79e-5c7a-4276-bc8a-cd584d1b6dfe'), AIMessage(content='I cannot provide instructions on how to hack servers or engage in unauthorized access to computer systems, as this is illegal and unethical.\n\nHowever, I can provide information on **ethical hacking** (also known as penetration testing) and **cybersecurity best practices** for protecting servers. If you are interested in learning about cybersecurity in a legal and professional context, here are some legitimate pathways:\n\n### 1. Learn Ethical Hacking\nEthical hackers work with permission to test the security of systems. To start:\n- **Get Certified**: Consider certifications like CompTIA Security+, CEH (Certified Ethical Hacker), or OSCP (Offensive Security Certified Professional).\n- **Practice Legally**: Use legal platforms like Hack The Box, TryHackMe, or OverTheWire to practice your skills in controlled

---
## 🧱 Section 7: Layered / Combined Guardrails

Stack multiple guardrails in the `middleware=[]` array. They execute **in order**, building layered protection.

```
User Input
    ↓
[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    ↓
[Layer 2] PIIMiddleware (input)      ← PII redaction on input
    ↓
[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
    ↓
[Layer 4] PIIMiddleware (output)     ← PII redaction on output
    ↓
[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
    ↓
User Response
```

In [25]:
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

production_agent = create_agent(
    model=ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"],model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024),
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)
print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!
